# Data Simulation Tool (Quick demo)

## Case : Online shop checkout page (one-page checkout)
Case description - Assume we have an online shop and we would like to test whether a simplified **one-page checkout** leads to more completed purchases than the current multi-step checkout. Users are randomly shown one of the two versions.

**How the tool generates the data for this case**

- **Customers**: a random number of customers between 40,000 and 60,000.
- **Checkout design** (`checkout_design`): each customer is randomly assigned to version A (old design) or B(new design).
- **Default customer details** (generated automatically):
    - `user_reg_name`: random names
    - `age`: between 18 and 74
    - `gender`: Male, Female, Non-binary or missing
    - `user_country_of_origin`: country code, with `NL` and `nl` both used, and some missing
    - `user_registration_date`: between 2006 and 2026
    - `device`: full browser text (user agent), not a clean device type
- **Our custom details** (random, evenly spread):
    - `previous_orders`: any value between 0 and 50, with 2 decimals
    - `has_discount_code`: 0 or 1, about 50/50
    - `traffic_source`: search, social or email, about equally often
- **Purchase** (`purchase_completed`): each customer gets a score, which is turned into a chance of buying (0 = no, 1 = yes).
- **Files**: `treatment_vars.pkl` (`id` + checkout design) and `outcome_and_control_vars.pkl` (`ID` + purchase + all details).

## Demo outline: Generating data for the A/B test 

1. **Open the tool**: [simulatedata.applikuapp.com](https://simulatedata.applikuapp.com)
    - Let's open the tool and look at the generation parameters.

2. **Define the A/B test variable**: `checkout_design` (version A = current, version B = new design)
    - We only type the name; the tool creates version A and version B and randomly assigns customers.
    - This is our independent variable: the one thing we change between the two groups.
    - The tool doesn't know what A and B mean. You decide that and describe it in your report.

3. **Define the dependent variable**: `purchase_completed`, binary (0/1)
    - This is the outcome we want to measure: did the customer complete the purchase? (1 = yes, 0 = no)
    - The tool offers two types: binary (yes/no) or numeric (a range, e.g. time on page 0–300 seconds).
    - Choose the type that fits your question. It also determines the analysis later: binary → logistic regression, numeric → linear regression.

4. **Add custom covariates** (2–3 max):
    - `previous_orders`: numeric (0 to 50)
    - `has_discount_code`: binary (0/1)
    - `traffic_source`: categorical (search, social, email)
    - Covariates are control variables: we don't change them, they describe the customers.
    - Why add them? They help explain who buys anyway (e.g. loyal customers), so the effect of the checkout design is estimated more precisely.
    - Choose things known *before* the customer sees the checkout, so the design cannot affect them.
    - Pick realistic ranges and categories for your case.

5. **Check the default variables**: name, age, gender, country, registration date, device type (generated automatically)
    - These come for free, so don't add them again as custom covariates.
    - You don't need all of them: choose the ones relevant to your case (e.g. device type may matter for a checkout page).
    - Name is just an identifier, so leave it out of the analysis.
    - Registration date can be turned into something useful, like how long someone has been a customer.

6. **Simulate data** and check the preview
    - Click "Simulate data" and look at the preview before downloading.
    - Check that the treatment, outcome and covariates look as you defined them (names, ranges, categories).
    - If something is wrong, fix the settings now. It's easier than fixing it after downloading.

7. **Download data**: save the ZIP once (a new download gives a new dataset). It contains two files: ID + treatment, and ID + outcome + covariates
    -  Every download creates a new dataset, so you can't get the same data back.
    - Use this one dataset for your whole report.
    - The treatment is in a separate file, so we only see who got which checkout after merging.

8. **Merge the two files** on the ID variable
    - Each customer has the same ID in both files, so we match the rows by ID.
    - Check the number of rows before and after merging. They should make sense.
    - Watch for IDs that appear in only one file: these are a cleaning issue for the next step.

9. **Pre-process**: check and fix duplicates, missing values, out-of-range values, category labels, data types; recode A = 0 / B = 1

10. **Analysis**: compare purchase rates A vs B, then a logistic regression with covariates

## Load, check and merge the data

In [42]:
# checking the path
!ls downloaded_data/ab_test_data/

outcome_and_control_vars.pkl treatment_vars.pkl


In [58]:
import pandas as pd

df_treatment = pd.read_pickle("downloaded_data/ab_test_data/treatment_vars.pkl")   
df_outcome = pd.read_pickle("downloaded_data/ab_test_data/outcome_and_control_vars.pkl")       

In [59]:
# checking the row size for the treatment data
len(df_treatment),len(df_treatment.columns)

(41785, 2)

In [60]:
# checking the row size outcome data
len(df_outcome),len(df_outcome.columns)

(41785, 11)

In [61]:
df_treatment.head(2)

,id,checkout_design
0,1,version_B
1,2,version_B


In [62]:
df_outcome.head(2)

,ID,purchase_completed,previous_orders,has_discount_code,traffic_source,user_reg_name,age,gender,user_country_of_origin,user_registration_date,device
0,1,0,10.18,1,email,Alicia Garcia,73,Female,nl,2013-09-26,Mozilla/5.0 (compatible; MSIE 5.0; Windows NT ...
1,2,1,21.19,0,email,Paige Ayers,66,Male,nl,2007-05-30,Mozilla/5.0 (Macintosh; PPC Mac OS X 10_5_2; r...


In [63]:
# Check the stat to see how the tool generated - tells the distribution
pd.options.display.float_format = "{:.2f}".format
df_outcome.describe()

,ID,purchase_completed,previous_orders,has_discount_code,age,user_registration_date
count,41785.00,41785.00,41785.00,41785.00,41785.00,41785
mean,20893.00,0.50,25.01,0.50,46.01,2015-12-25 20:30:50.960871168
min,1.00,0.00,0.00,0.00,18.00,2006-01-01 00:00:00
25%,10447.00,0.00,12.41,0.00,32.00,2010-12-27 00:00:00
50%,20893.00,0.00,24.93,1.00,46.00,2015-12-21 00:00:00
75%,31339.00,1.00,37.62,1.00,60.00,2020-12-03 00:00:00
max,41785.00,1.00,50.00,1.00,74.00,2025-12-26 00:00:00
std,12062.43,0.50,14.50,0.50,16.42,NaN


In [64]:
# checking available columns
list(df_treatment.columns)

['id', 'checkout_design']

In [65]:
# checking available columns
list(df_outcome.columns)

['ID',
 'purchase_completed',
 'previous_orders',
 'has_discount_code',
 'traffic_source',
 'user_reg_name',
 'age',
 'gender',
 'user_country_of_origin',
 'user_registration_date',
 'device']

### Merging 

In [66]:
df_outcome["ID"].equals(df_treatment["id"])

True

In [38]:
# There is a difference between the id column name 
# Option-1
df_outcome1 = df_outcome.rename(columns={"ID": "id"})
df = pd.merge(df_treatment, df_outcome1, on="id", how="outer", indicator=True)